# 02. Демонстрация поиска через MCP

Этот ноутбук показывает как агент обращается к базе знаний
через MCP-протокол, а не напрямую через SDK.

Цепочка:
запрос → MCP-клиент → mcp-server-qdrant → Qdrant → top-k чанков с метаданными

Это и есть протокольная граница из задания:
поиск идёт не через langchain-qdrant напрямую,
а через стандартный MCP-инструмент.

In [1]:
import os
from pathlib import Path
from dotenv import load_dotenv

load_dotenv(Path(".env"))

GIGACHAT_CREDENTIALS = os.getenv("GIGACHAT_CREDENTIALS")
GIGACHAT_SCOPE       = os.getenv("GIGACHAT_SCOPE", "GIGACHAT_API_PERS")
QDRANT_PATH          = os.getenv("QDRANT_PATH", "./qdrant_storage")
QDRANT_COLLECTION    = os.getenv("QDRANT_COLLECTION", "dnd_phb")

# Абсолютный путь нужен для MCP-сервера — он запускается как отдельный процесс
QDRANT_ABS_PATH = str(Path(QDRANT_PATH).resolve())

print("GIGACHAT_CREDENTIALS:", "ОК: задан" if GIGACHAT_CREDENTIALS else "НЕ ОК: не найден")
print("QDRANT_ABS_PATH:", QDRANT_ABS_PATH)
print("QDRANT_COLLECTION:", QDRANT_COLLECTION)

GIGACHAT_CREDENTIALS: ОК: задан
QDRANT_ABS_PATH: /Users/alexandrerdenko/Desktop/ВШЭ/Учеба/Магистратура_ПМИ/1 курс/ИИ Агенты/hw_4/qdrant_storage
QDRANT_COLLECTION: dnd_phb


## Подключаем MCP-сервер

MultiServerMCPClient запускает mcp-server-qdrant как дочерний процесс (stdio транспорт).
Сервер получает путь к БД и название коллекции через переменные окружения.
Затем мы получаем список инструментов которые сервер публикует.

In [8]:
from langchain_mcp_adapters.client import MultiServerMCPClient

mcp_config = {
    "qdrant-dnd": {
        "transport": "streamable_http",
        "url": "http://127.0.0.1:8000/mcp",
    }
}

client = MultiServerMCPClient(mcp_config)
print("MCP-клиент создан")
print("Транспорт: streamable_http")
print("URL: http://127.0.0.1:8000/mcp")

MCP-клиент создан
Транспорт: streamable_http
URL: http://127.0.0.1:8000/mcp


In [9]:
# Получаем список инструментов которые публикует MCP-сервер
tools = await client.get_tools()

print(f"Инструментов получено: {len(tools)}\n")
for tool in tools:
    print(f"  Имя        : {tool.name}")
    print(f"  Описание   : {tool.description[:100]}")
    print()

Инструментов получено: 2

  Имя        : qdrant-find
  Описание   : Look up memories in Qdrant. Use this tool when you need to: 
 - Find memories by their content 
 - A

  Имя        : qdrant-store
  Описание   : Keep the memory for later use, when you are asked to remember something.



## Демонстрация поиска через MCP-инструмент

Вызываем qdrant-find напрямую — без агента.
Это ручной вызов MCP-инструмента.
Видим что инструмент принимает запрос и возвращает результаты из Qdrant.

In [10]:
# Найдём инструмент поиска по имени
search_tool = None
for tool in tools:
    if tool.name == "qdrant-find":
        search_tool = tool
        break

print(f"Инструмент найден: {search_tool.name}")
print(f"Описание: {search_tool.description}")

Инструмент найден: qdrant-find
Описание: Look up memories in Qdrant. Use this tool when you need to: 
 - Find memories by their content 
 - Access memories for further analysis 
 - Get some personal information about the user


In [11]:
# Ручной вызов MCP-инструмента поиска
query = "какой урон наносит огненный шар"

print(f"Запрос: '{query}'")

result = await search_tool.ainvoke({"query": query})
print(result)

Запрос: 'какой урон наносит огненный шар'
[{'type': 'text', 'text': '[\n  "Results for the query \'какой урон наносит огненный шар\'",\n  "<entry><content>верки характеристик, сделанных для оценки или \\nисследования мелких и высокодетализированных \\nпредметов. \\nФакел. Факел горит 1 час, испуская яркий свет \\nв пределах 20 футов и тусклый свет в пределах \\nещё 20 футов. Если вы совершаете рукопашную \\nатаку горящим факелом и попадаете, он причи-\\nняет урон огнём 1. \\nФокусировка друидов. Фокусировкой друида мо-\\nжет быть веточка омелы или падуба, палочка или \\nскипетр из тиса или другого дерева, посох, создан-</content><metadata></metadata></entry>",\n  "<entry><content>существа в пределах 15-футового конуса должны \\nсовершить спасброски Ловкости. Существо полу-\\nчает урон огнём 3к6 в случае провала и половину \\nэтого урона в случае успеха. \\nПламя поджигает все горючие предметы, ни-\\nкем не несомые и не носимые. \\nНа больших уровнях: Если вы накладываете \\nэто заклина

## Парсим результаты MCP-поиска

Инструмент возвращает сырой текст. Разбираем его и показываем
в читаемом формате с метаданными.

In [12]:
import json
import re

async def mcp_search(query: str, k: int = 3):
    """Поиск через MCP-инструмент с красивым выводом."""
    raw = await search_tool.ainvoke({"query": query})
    
    # Извлекаем JSON из результата
    text = raw[0]["text"] if isinstance(raw, list) else raw
    entries = json.loads(text)
    
    print(f"Запрос: '{query}'")
    print(f"{'='*60}")
    
    # Первый элемент — заголовок, остальные — результаты
    results = entries[1:k+1]
    for i, entry in enumerate(results, 1):
        # Извлекаем текст из <entry><content>...</content>
        content_match = re.search(r'<content>(.*?)</content>', entry, re.DOTALL)
        text_content = content_match.group(1).strip() if content_match else entry
        
        print(f"\n[{i}] {text_content[:300]}")
        print(f"     {'—'*50}")
    
    return results

# Тестовый запрос
results = await mcp_search("какой урон наносит огненный шар", k=3)

Запрос: 'какой урон наносит огненный шар'

[1] верки характеристик, сделанных для оценки или 
исследования мелких и высокодетализированных 
предметов. 
Факел. Факел горит 1 час, испуская яркий свет 
в пределах 20 футов и тусклый свет в пределах 
ещё 20 футов. Если вы совершаете рукопашную 
атаку горящим факелом и попадаете, он причи-
няет урон о
     ——————————————————————————————————————————————————

[2] существа в пределах 15-футового конуса должны 
совершить спасброски Ловкости. Существо полу-
чает урон огнём 3к6 в случае провала и половину 
этого урона в случае успеха. 
Пламя поджигает все горючие предметы, ни-
кем не несомые и не носимые. 
На больших уровнях: Если вы накладываете 
это заклинание
     ——————————————————————————————————————————————————

[3] 226
ЧАСТЬ 3 : ЗАКЛИНАНИЯ
 
 
Огонь причиняет урон предметам и воспламе-
няет горючие предметы, которые никто не несёт 
и не носит. 
На больших уровнях: Если вы накладываете 
это заклинание, используя ячейку 8 уровня или 
выше, ба

In [13]:
# Ещё несколько запросов для проверки
test_queries = [
    "требования для мультиклассирования паладина",
    "как работает вдохновение барда",
    "что такое спасбросок от смерти",
]

for query in test_queries:
    await mcp_search(query, k=2)
    print()

Запрос: 'требования для мультиклассирования паладина'

[1] умения Использование заклинаний; и 
мультиклассирование
     ——————————————————————————————————————————————————

[2] В этой главе представлены два опциональных 
набора правил по индивидуальной настройке персо-
нажей: мультиклассирование и черты. Мультикласси-
рование позволяет комбинировать классы, а черты 
можно брать вместо увеличения характеристик при 
получении новых уровней. Будут ли доступны эти оп-
ции в ва
     ——————————————————————————————————————————————————

Запрос: 'как работает вдохновение барда'

[1] жет помочь в защите родной деревни, когда вы 
сопротивляетесь вредоносному заклинанию вол-
шебника, который напал на неё. 
 
КОГДА ДАЁТСЯ ВДОХНОВЕНИЕ 
Мастер может даровать вам вдохновение по не-
скольким причинам. Обычно вдохновение даётся 
за хороший отыгрыш черты характера, слабости 
или привязан
     ——————————————————————————————————————————————————

[2] ИС
ПОЛЬЗОВАНИЕ ВДОХНОВЕНИЯ 
Если у вас е
сть вдохновение, 

## Итог: поиск через MCP работает

Продемонстрировано:
- MCP-сервер: mcp-server-qdrant (транспорт: streamable-http)
- Инструмент: qdrant-find
- Коллекция: dnd_phb_mcp (3182 чанков из PHB на русском)
- Модель эмбеддингов: sentence-transformers/paraphrase-multilingual-mpnet-base-v2

Агент вызывает qdrant-find с текстовым запросом и получает
релевантные фрагменты из книги игрока D&D.

Метаданные (chunk_id, document_id, source, page) хранятся
в Qdrant и доступны через прямой поиск (см. 01_ingest.ipynb).